# Parte 0

## Contexto

Antes de começar, pesquisei o Programa Pequenos Cariocas para entender o problema real por trás do desafio, não apenas os dados.

O programa integra ações de saúde, educação e assistência social para crianças de 0 a 6 anos e gestantes em vulnerabilidade no Rio. Uma das ações é o Cartão da Primeira Infância Carioca (PIC), que distribui R$ 200 mensais para famílias com filhos de até 4 anos via cruzamento automático com o CadÚnico, um exemplo prático de como integração de dados entre secretarias viabiliza política pública.

O desafio usa os dados do 1746 para observar essa demanda na prática. Criada em 2011, a Central reúne centenas de serviços municipais e funciona como linha direta entre a Prefeitura e o cidadão, por telefone, app, portal e WhatsApp. Cada chamado recebe um prazo e um protocolo, e o índice de resolução faz parte do acordo de resultados com todos os órgãos da prefeitura.

Esse último ponto importa para a modelagem: o schema da tabela registra o status "fechado com solução", mas quem define isso é o operador do sistema, não o cidadão. Um chamado pode ser encerrado com essa classificação após uma resolução provisória. Na prática, a variável-alvo da Parte 2 será construída com base nesse status, mas com essa limitação documentada.

**Referências**
- [Programa Pequenos Cariocas, Prefeitura do Rio](https://prefeitura.rio/cidade/programa-pequenos-cariocas-cartao-da-primeira-infancia-vai-garantir-a-seguranca-alimentar-das-criancas/) (12/10/2025)
- [Tecnologia da IplanRio no programa](https://iplanrio.prefeitura.rio/noticias/pequenos-cariocas-tecnologia-da-iplanrio-apoia-a-gestao-e-a-seguranca-dos-dados-no-novo-programa-da-prefeitura-do-rio/) (16/10/2025)
- [Cartão PIC, três meses de operação, SMAS](https://assistenciasocial.prefeitura.rio/noticias/cartao-da-primeira-infancia-carioca-completa-tres-meses-de-combate-a-inseguranca-alimentar/) (12/01/2026)
- [Central 1746, dez anos de linha direta com os cariocas, Prefeitura do Rio](https://prefeitura.rio/cidade/central-1746-comemora-dez-anos-a-servico-dos-cariocas/) (23/03/2021)
- [Schema da tabela chamado, prefeitura-rio/queries-datario](https://github.com/prefeitura-rio/queries-datario)

In [1]:
import pandas as pd
from src.collectors import DataPipeline

In [2]:
pipeline = DataPipeline(
    billing_project_id="desafio-pic-ds-494517",
    start_date="2023-01-01",
    end_date="2024-12-31",
)

dados = pipeline.run()

=== Coletando chamados do 1746 ===
[ChamadosCollector] Carregando de C:\Users\marie\Documents\desafio-cientista-dados-senior-cidadaos-vulneraveis\data\raw\chamados.parquet

=== Coletando dados mestres ===
[DadosMestresCollector] Carregando de C:\Users\marie\Documents\desafio-cientista-dados-senior-cidadaos-vulneraveis\data\raw\bairro.parquet
[DadosMestresCollector] Carregando de C:\Users\marie\Documents\desafio-cientista-dados-senior-cidadaos-vulneraveis\data\raw\area_planejamento.parquet
[DadosMestresCollector] Carregando de C:\Users\marie\Documents\desafio-cientista-dados-senior-cidadaos-vulneraveis\data\raw\regiao_administrativa.parquet
[DadosMestresCollector] Carregando de C:\Users\marie\Documents\desafio-cientista-dados-senior-cidadaos-vulneraveis\data\raw\subprefeitura.parquet

=== Coletando dados climáticos ===
[OpenMeteoCollector] Carregando de C:\Users\marie\Documents\desafio-cientista-dados-senior-cidadaos-vulneraveis\data\raw\clima.parquet

=== Coletando feriados ===
[Public

## Análise Exploratória

### Geral

In [3]:
df = dados["chamados"]

print(df.shape)
print(df.dtypes)

(2792446, 34)
id_chamado                                          str
id_origem_ocorrencia                                str
data_inicio                              datetime64[us]
data_fim                                 datetime64[us]
id_bairro                                           str
id_territorialidade                                 str
id_logradouro                                       str
numero_logradouro                                 Int64
id_unidade_organizacional                           str
nome_unidade_organizacional                         str
id_unidade_organizacional_mae                       str
unidade_organizacional_ouvidoria                    str
categoria                                           str
id_tipo                                             str
tipo                                                str
id_subtipo                                          str
subtipo                                             str
status                            

- `data_particao` é do tipo dbdate, vai precisar de conversão.
- `updated_at` está como str, deveria ser datetime.
- `dentro_prazo` está como str, deveria ser booleano.

### Nulos

In [4]:
nan_pct = df.isnull().sum() / len(df) * 100
nan_pct = nan_pct[nan_pct > 0].sort_values(ascending=False)
print(nan_pct.round(2))

data_alvo_diagnostico            97.07
data_real_diagnostico            96.80
justificativa_status             93.81
latitude                         55.25
longitude                        55.25
numero_logradouro                36.53
id_bairro                        31.52
id_territorialidade              31.52
id_logradouro                    31.52
data_alvo_finalizacao            21.83
tempo_prazo                      21.81
prazo_unidade                    21.81
prazo_tipo                       21.81
data_fim                          1.51
id_unidade_organizacional_mae     0.00
dtype: float64


As variáveis a seguir só devem ser preenchidas tipos específicos de chamado, por conta do altíssima quantidade de nulos:
- data_alvo_diagnostico 
- data_real_diagnostico 
- justificativa_status


Sobre logo em seguida termos latitude e longitude com mais da metade de nulos, pode ser mais útil trabalhar com `id_bairro`, que tem 32% de nulos, do que insistir na coordenada geográfica com 55% ausente.

In [5]:
for col in ["status", "situacao", "tipo_situacao", "dentro_prazo", "categoria"]:
    print(f"\n{col} ({df[col].nunique()} únicos):")
    print(df[col].value_counts().head(10))


status (10 únicos):
status
Fechado com solução                 1177879
Fechado com informação              1074748
Não constatado                       188379
Sem possibilidade de atendimento     180358
Fechado com providências             136896
Aberto                                16630
Pendente                               8416
Cancelado                              5789
Em Andamento                           2720
Fechado de Ofício                       631
Name: count, dtype: int64

situacao (2 únicos):
situacao
Encerrado        2764691
Não Encerrado      27755
Name: count, dtype: int64

tipo_situacao (5 únicos):
tipo_situacao
Atendido parcialmente    1211644
Atendido                 1177879
Não constatado            188379
Não atendido              186778
Andamento                  27766
Name: count, dtype: int64

dentro_prazo (5 únicos):
dentro_prazo
A Vencer (No Prazo)         1597754
Não Calculado                607723
Em Vencimento (No Prazo)     302328
Vencido             

`tipo_situacao` explica o status. "Atendido parcialmente" tem 1.2M de registros contra 1.17M de "Fechado com solução". Isso sugere que "Fechado com informação" mapeia para "Atendido parcialmente": o chamado foi encerrado, mas sem uma solução completa.

In [6]:
print(pd.crosstab(df["status"], df["tipo_situacao"]))

tipo_situacao                     Andamento  Atendido  Atendido parcialmente  \
status                                                                         
Aberto                                16630         0                      0   
Cancelado                                 0         0                      0   
Em Andamento                           2720         0                      0   
Fechado com informação                    0         0                1074748   
Fechado com providências                  0         0                 136896   
Fechado com solução                       0   1177879                      0   
Fechado de Ofício                         0         0                      0   
Não constatado                            0         0                      0   
Pendente                               8416         0                      0   
Sem possibilidade de atendimento          0         0                      0   

tipo_situacao                     Não a

- "Fechado com solução" mapeia 1:1 para "Atendido"
- "Fechado com informação" e "Fechado com providências" mapeiam para "Atendido parcialmente"
- "Não constatado" mapeia para "Não constatado"
- "Sem possibilidade de atendimento" e "Cancelado" mapeiam para "Não atendido"
- Chamados em aberto ("Aberto", "Em Andamento", "Pendente") mapeiam para "Andamento"

"Não Calculado" em `dentro_prazo` corresponde aos 22% de `data_alvo_finalizacao` ausente. São chamados sem prazo definido, provavelmente de categorias como "Informações" e "Elogio" que não geram serviço.

In [7]:
print(pd.crosstab(df["dentro_prazo"], df["data_alvo_finalizacao"].isnull(), 
                  margins=True))

data_alvo_finalizacao       False    True      All
dentro_prazo                                      
A Vencer (No Prazo)       1597754       0  1597754
Em Vencimento (No Prazo)   302328       0   302328
Não Calculado                   0  607723   607723
Pendente de Programação         0    1796     1796
Vencido                    282845       0   282845
All                       2182927  609519  2792446


Todo chamado com `dentro_prazo == "Não Calculado"` tem `data_alvo_finalizacao` nula, e vice-versa: 609.519 registros sem prazo definido.

In [8]:
df[df["dentro_prazo"] == "Não Calculado"]["categoria"].value_counts()

categoria
Informações    579696
Crítica         16382
Reclamação       3739
Serviço          3716
Elogio           2640
Sugestão         1550
Name: count, dtype: int64

"Informações" domina com 95% dos chamados sem prazo, o que confirma a hipótese principal. Mas há 3.716 chamados de "Serviço" sem prazo definido, o que é inesperado, serviços deveriam ter prazo. Vale investigar depois se são casos especiais ou inconsistências nos dados.

### Tempo

In [9]:
print(df["data_inicio"].dt.year.value_counts().sort_index())
print(f"\nData mais antiga: {df['data_inicio'].min()}")
print(f"Data mais recente: {df['data_inicio'].max()}")

data_inicio
2023    1348995
2024    1443451
Name: count, dtype: int64

Data mais antiga: 2023-01-01 00:23:44
Data mais recente: 2024-12-31 23:52:11


O período está exatamente como esperado: 2023 e 2024 completos. O volume cresceu cerca de 7% de um ano para o outro (1.35M para 1.44M), o que pode ser sazonalidade, crescimento real de demanda ou mudança no registro.

In [10]:
tempo_resolucao = (df["data_fim"] - df["data_inicio"]).dt.days
print(tempo_resolucao.describe())
print(f"\nResolvidos em até 7 dias: {(tempo_resolucao <= 7).sum():,} ({(tempo_resolucao <= 7).mean()*100:.1f}%)")

count    2.750327e+06
mean     1.492495e+01
std      7.072215e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      6.000000e+00
max      1.179000e+03
dtype: float64

Resolvidos em até 7 dias: 2,197,378 (78.7%)


78.7% dos chamados resolvidos em até 7 dias representa um desbalanceamento significativo de classes: o modelo vai tender a prever sempre a classe majoritária, e isso precisa entrar nas decisões de modelagem com estratégias como `class_weight='balanced'` ou reamostragem. A mediana de 0 dias reforça o ponto: mais de metade dos chamados é encerrada no mesmo dia em que é aberta, provavelmente os de "Informações" com resolução imediata, o que sugere filtrar ou separar essas categorias antes de modelar.

## Conclusão

A exploração revela três pontos que vão orientar as próximas etapas.

O primeiro é sobre a construção da variável-alvo: o dataset tem o campo `dentro_prazo`, mas ele mede conformidade com o prazo administrativo de cada serviço, que varia por tipo. Para o objetivo da Parte 2, a variável-alvo precisa ser construída a partir de `data_inicio` e `data_fim`, restrita aos chamados com `status == 'Fechado com solução'`.

O segundo é sobre desbalanceamento: 78.7% dos chamados são encerrados em até 7 dias, com mediana de 0 dias. Parte disso é estrutural, chamados de "Informações" têm resolução imediata, representando o grosso dos 609k casos sem prazo definido. Incluí-los no dataset de modelagem sem tratamento distorce a variável-alvo.

O terceiro é sobre cobertura geoespacial: latitude e longitude ausentes em 55% dos registros limitam a granularidade da análise espacial. A alternativa é trabalhar com `id_bairro`, que tem 32% de nulos, aceitável para análises por território.